# 06. Hierarchical RL Options Feudal Her and POMDP Active Inference

- **Category:** 08_learning_paradigms_rl_and_alignment / 02_reinforcement_learning_and_world_models
- **Lineage & Citations:** Sutton, Precup & Singh (1999 Options Framework); Bacon, Harb & Precup (2017 Option-Critic Architecture); Vezhnevets et al. (2017 FeUdal Networks); Andrychowicz et al. (2017 Hindsight Experience Replay); Friston (2006, 2010 Active Inference & Free Energy Principle).
- **Purpose:** First-principles mathematical implementation and empirical verification of hierarchical reinforcement learning and variational active inference: Hindsight Experience Replay (HER) sparse goal relabeling, FeUdal Networks latent directional cosine rewards, Option-Critic termination gradient with deliberation cost, and Variational Free Energy for POMDP belief tracking.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [05. Latent World Models Rssm Dreamer Tdmpc and Rnd Curiosity](05_latent_world_models_rssm_dreamer_tdmpc_and_rnd_curiosity.ipynb) $\to$ `🔗 Downstream Syntheses:` [07. Multi-Agent RL Qmix Mappo CFR and Agentic Reasoning Loops](07_multi_agent_rl_qmix_mappo_cfr_and_agentic_reasoning_loops.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to Use**  
> Never omit the deliberation cost $\eta > 0$ when training Option-Critic termination functions. Without $\eta$, options either degenerate into primitive 1-step actions ($\beta_\omega \to 1$) or freeze into indefinite inertia ($\beta_\omega \to 0$). In Hindsight Experience Replay (HER), never use an uncalibrated goal proximity threshold $\delta$; setting $\delta$ too wide rewards arbitrary inaccurate states, while setting it too tight creates zero gradient.

## Mathematical Foundations & Governing Equations

### 1. Hindsight Experience Replay (HER)
$$
r(s, g) = \begin{cases} 1.0 & \text{if } \|s - g\|_2 \le \delta \\ 0.0 & \text{otherwise} \end{cases}
$$

### 2. FeUdal Networks Latent Directional Reward
$$
r_t^W = \frac{(\mathbf{s}_{t+c} - \mathbf{s}_t) \cdot \mathbf{g}_t}{\max \left( \|\mathbf{s}_{t+c} - \mathbf{s}_t\|_2 \|\mathbf{g}_t\|_2, \, \epsilon \right)}
$$

### 3. Option-Critic Termination Gradient with Deliberation Cost
$$
\mathcal{L}_{\text{term}}(\theta) = \frac{1}{B} \sum_{i=1}^B \beta_\omega(s_i) \left[ Q_\Omega(s_i, \omega) - V(s_i) + \eta \right]
$$

### 4. POMDP Active Inference Variational Free Energy
$$
\mathcal{F} = D_{\text{KL}}\left( q_\phi(s) \,\|\, p(s) \right) + \frac{1}{2} \sum_{d=1}^{D_o} \left[ \frac{(o_d - \hat{o}_d)^2}{\sigma_o^2} + \ln(2\pi \sigma_o^2) \right]
$$

### Asymptotic Time & Memory Complexity
- HER Goal Relabeling: $\mathcal{O}(B \cdot D)$ time and memory.
- FeUdal Directional Reward: $\mathcal{O}(B \cdot D)$ time and memory.
- Option-Critic Termination Loss: $\mathcal{O}(B)$ time and memory.
- POMDP Variational Free Energy: $\mathcal{O}(B \cdot (D_s + D_o))$ time and memory.

In [ ]:
# Setup and seed verification environment
import torch
import torch.nn.functional as F
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)

# Locate the golden checkpoint
CKPT_REL = Path("weights/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/06_hierarchical_rl_options_feudal_her_and_pomdp_active_inference.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
print(f"Loaded {len(weights)} golden tensors on {env.device}")

In [ ]:
def linear_anchor(x: torch.Tensor, w: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    return x @ w + b

def her_goal_relabeling(
    states: torch.Tensor,
    goals: torch.Tensor,
    threshold: float = 0.1,
) -> torch.Tensor:
    dists = torch.norm(states - goals, p=2, dim=-1)
    return (dists <= threshold).float()

def feudal_directional_reward(
    delta_s: torch.Tensor,
    subgoal_g: torch.Tensor,
    eps: float = 1e-6,
) -> torch.Tensor:
    dot_product = torch.sum(delta_s * subgoal_g, dim=-1)
    norm_delta = torch.norm(delta_s, p=2, dim=-1)
    norm_g = torch.norm(subgoal_g, p=2, dim=-1)
    denom = torch.clamp(norm_delta * norm_g, min=eps)
    return dot_product / denom

def option_critic_termination_loss(
    beta_omega: torch.Tensor,
    q_omega: torch.Tensor,
    v_s: torch.Tensor,
    deliberation_cost: float = 0.01,
) -> torch.Tensor:
    advantage = q_omega - v_s + deliberation_cost
    return torch.mean(beta_omega * advantage)

def active_inference_free_energy(
    q_mean: torch.Tensor,
    q_log_std: torch.Tensor,
    p_mean: torch.Tensor,
    p_log_std: torch.Tensor,
    obs: torch.Tensor,
    recon: torch.Tensor,
    obs_noise_var: float = 1.0,
) -> torch.Tensor:
    var_q = torch.exp(2.0 * q_log_std)
    var_p = torch.exp(2.0 * p_log_std)
    diff = q_mean - p_mean
    kl_element = (var_q + diff ** 2) / var_p - 1.0 + 2.0 * (p_log_std - q_log_std)
    kl_div = 0.5 * torch.sum(kl_element, dim=-1)
    sq_err = (obs - recon) ** 2
    obs_dim = obs.shape[-1]
    log_2pi_var = torch.log(torch.tensor(2.0 * 3.141592653589793 * obs_noise_var, device=obs.device))
    recon_nll = 0.5 * (torch.sum(sq_err, dim=-1) / obs_noise_var + obs_dim * log_2pi_var)
    return kl_div + recon_nll

In [ ]:
# Multi-track numerical parity assertions
y = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
err_anchor = float(torch.max(torch.abs(y - weights["test_output"])))
assert err_anchor < 1e-5, f"Linear anchor parity error: {err_anchor:.2e}"

her_rew = her_goal_relabeling(weights["states"], weights["goals"], threshold=0.1)
err_her = float(torch.max(torch.abs(her_rew - weights["her_rewards_ref"])))
assert err_her < 1e-5, f"HER rewards parity error: {err_her:.2e}"

feudal_rew = feudal_directional_reward(weights["delta_s"], weights["subgoal_g"])
err_feudal = float(torch.max(torch.abs(feudal_rew - weights["feudal_rewards_ref"])))
assert err_feudal < 1e-5, f"FeUdal rewards parity error: {err_feudal:.2e}"

opt_loss = option_critic_termination_loss(weights["beta_omega"], weights["q_omega"], weights["v_s"], deliberation_cost=0.01)
err_opt = float(torch.max(torch.abs(opt_loss - weights["option_loss_ref"])))
assert err_opt < 1e-5, f"Option-Critic termination loss parity error: {err_opt:.2e}"

fe = active_inference_free_energy(
    weights["q_mean"],
    weights["q_log_std"],
    weights["p_mean"],
    weights["p_log_std"],
    weights["obs"],
    weights["recon"],
    obs_noise_var=1.0,
)
err_fe = float(torch.max(torch.abs(fe - weights["free_energy_ref"])))
assert err_fe < 1e-5, f"Active inference free energy parity error: {err_fe:.2e}"

print(f"All parity assertions passed successfully (max error {max(err_anchor, err_her, err_feudal, err_opt, err_fe):.2e})")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

b_her = benchmark_fn(
    lambda: her_goal_relabeling(weights["states"], weights["goals"]),
    iters=100,
    name="HER Goal Relabeling",
)
b_feudal = benchmark_fn(
    lambda: feudal_directional_reward(weights["delta_s"], weights["subgoal_g"]),
    iters=100,
    name="FeUdal Directional Reward",
)
b_opt = benchmark_fn(
    lambda: option_critic_termination_loss(weights["beta_omega"], weights["q_omega"], weights["v_s"]),
    iters=100,
    name="Option-Critic Termination Loss",
)
b_fe = benchmark_fn(
    lambda: active_inference_free_energy(
        weights["q_mean"], weights["q_log_std"], weights["p_mean"], weights["p_log_std"], weights["obs"], weights["recon"]
    ),
    iters=100,
    name="POMDP Variational Free Energy",
)

print(format_benchmark_table([b_her, b_feudal, b_opt, b_fe]))

In [ ]:
# Failure Mode Contrast: Option-Critic with vs without Deliberation Cost
# Suppose the current option value exactly equals state value Q_omega = V_s = 2.0
beta_param = torch.tensor([0.5], requires_grad=True)
q_val = torch.tensor([2.0])
v_val = torch.tensor([2.0])

# Case 1: Without Deliberation Cost (deliberation_cost = 0.0)
# Advantage is exactly zero: gradient w.r.t beta vanishes, creating option thrashing
loss_no_cost = beta_param * (q_val - v_val + 0.0)
grad_no_cost = torch.autograd.grad(loss_no_cost, beta_param, retain_graph=True)[0].item()

# Case 2: With Deliberation Cost (deliberation_cost = 0.05)
# Positive deliberation penalty creates a clear gradient driving termination to 0 (persist option!)
loss_with_cost = beta_param * (q_val - v_val + 0.05)
grad_with_cost = torch.autograd.grad(loss_with_cost, beta_param)[0].item()

print(f"Termination gradient without deliberation cost: {grad_no_cost:.4f} (Zero gradient, option thrashing)")
print(f"Termination gradient with deliberation cost:    {grad_with_cost:.4f} (Explicit incentive to persist)")

assert abs(grad_no_cost - 0.0) < 1e-5, f"Expected 0 gradient without cost, got {grad_no_cost}"
assert abs(grad_with_cost - 0.05) < 1e-4, f"Expected 0.05 gradient with cost, got {grad_with_cost}"
print("Pathological failure mode contrast successfully verified")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Hindsight Experience Replay (HER)**: Counterfactual goal relabeling transforms 0%-success exploration failures into informative positive samples, dramatically accelerating sparse-reward RL.
2. **Directional Latent Invariants (FeUdal Networks)**: Cosine similarity rewards decouple spatial progress from scale, insulating low-level workers from high-level managerial drifting.
3. **Deliberation Regularization (Option-Critic)**: Explicit switching penalties $\eta$ are necessary to prevent rapid option toggling and discover temporally extended macro-behaviors.
4. **Variational Free Energy**: Unifies belief tracking in partially observable environments with goal-directed control under a single principle of bounding surprise.

---
### Cross-Domain Generalisation Matrix
- **Robotic Manipulation**: HER enables multi-joint robotic arms to master peg insertion and drawer opening with zero reward shaping.
- **Hierarchical Navigation**: Option-Critic discovers hallway transit and obstacle clearance sub-policies without domain-specific state partitioning.
- **Active Inference Diagnostics**: POMDP free energy minimization balances clinical test selection against treatment costs in medical diagnostic assistants.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [06_hierarchical_rl_options_feudal_her_and_pomdp_active_inference.md](../../../markdown/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/06_hierarchical_rl_options_feudal_her_and_pomdp_active_inference.md)
- **PyTorch 2.x Module:** [06_hierarchical_rl_options_feudal_her_and_pomdp_active_inference.py](../../../pytorch2/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/06_hierarchical_rl_options_feudal_her_and_pomdp_active_inference.py)
- **Functional JAX Module:** [06_hierarchical_rl_options_feudal_her_and_pomdp_active_inference.py](../../../jax/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/06_hierarchical_rl_options_feudal_her_and_pomdp_active_inference.py)
- **Native Rust Candle Track:** [06_hierarchical_rl_options_feudal_her_and_pomdp_active_inference.rs](../../../candle/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/06_hierarchical_rl_options_feudal_her_and_pomdp_active_inference.rs)
- **Hardware Kernel (CPU / WGSL):** [06_hierarchical_rl_options_feudal_her_and_pomdp_active_inference.rs](../../../kernels/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/06_hierarchical_rl_options_feudal_her_and_pomdp_active_inference.rs)
- **Golden SafeTensors:** [06_hierarchical_rl_options_feudal_her_and_pomdp_active_inference.safetensors](../../../weights/08_learning_paradigms_rl_and_alignment/02_reinforcement_learning_and_world_models/06_hierarchical_rl_options_feudal_her_and_pomdp_active_inference.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 05. Latent World Models Rssm Dreamer Tdmpc and Rnd Curiosity](05_latent_world_models_rssm_dreamer_tdmpc_and_rnd_curiosity.ipynb) | [Master Index](../../../README.md#08_learning_paradigms_rl_and_alignment) | [Next: 07. Multi-Agent RL Qmix Mappo CFR and Agentic Reasoning Loops](07_multi_agent_rl_qmix_mappo_cfr_and_agentic_reasoning_loops.ipynb) &rarr;